# Predicting Student Dropout and Academic Success

Use student demographic, academic, and socioeconomic data to identify students at risk of dropping out, remaining enrolled, or succeeding academically.

**Project Question:** How can an institution identify students who may need support early enough to improve student outcomes?

[Project Dataset](https://archive.ics.uci.edu/dataset/697/predict+students+dropout+and+academic+success)

**Models kept:** Random Forest and XGBoost (best performers). Other classifiers, PCA/t-SNE/UMAP, and the stacking ensemble were removed during cleanup.

**Focus of this version:** improve detection of the **Enrolled** class via consolidated feature engineering (before RFE), interaction features, and class-balancing techniques.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# For Kim:
import pandas as pd

# df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/iX class project/dropout_data.csv', sep=';')
# display(df)

In [ ]:
# For Hudson:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/data.csv', sep=';')
display(df)

In [ ]:
# For Jacari:
import pandas as pd

# df = pd.read_csv('/content/dropout_data.csv', sep=';')
# display(df)

## Data Cleaning

The dataset creators performed rigorous preprocessing (anomalies, outliers, missing values), so little cleaning is needed. We confirm below.

In [ ]:
# Confirm no missing values and review distributions
display(df.isnull().sum())
display(df.describe())

In [ ]:
print("Categorical columns:", df.select_dtypes(include='object').columns.tolist())
print("Numerical columns:", df.select_dtypes(include='number').columns.tolist())

## Data Exploration

Correlation of each numerical feature with dropout status.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Binary dropout flag for correlation analysis
df['Is_Dropout'] = (df['Target'] == 'Dropout').astype(int)

numerical_cols = df.select_dtypes(include='number').columns.tolist()
dropout_correlation = df[numerical_cols].corr()[['Is_Dropout']].sort_values(by='Is_Dropout', ascending=False)

plt.figure(figsize=(8, 15))
sns.heatmap(dropout_correlation, annot=True, cmap='coolwarm', fmt=".2f", cbar=True)
plt.title('Correlation of Numerical Features with Dropout Status')
plt.yticks(rotation=0)
plt.show()

## Class Balance Check

Confirm that **Enrolled** is the minority class — the root of the recall problem.

In [ ]:
print("Target counts:\n", df['Target'].value_counts())
print("\nTarget proportions:\n", df['Target'].value_counts(normalize=True).round(3))

## Feature Engineering (Round 1)

First-semester performance and financial-risk features.

In [ ]:
# 1. Approval rate (share of enrolled 1st-sem classes passed)
df["approval_rate_1st_sem"] = (
    df["Curricular units 1st sem (approved)"] /
    df["Curricular units 1st sem (enrolled)"]
).fillna(0)

# 2. Failed courses, 1st sem
df["failed_courses_1st_sem"] = (
    df["Curricular units 1st sem (enrolled)"] -
    df["Curricular units 1st sem (approved)"]
)

# 3. Average 1st-sem grade
df["avg_first_sem_grade"] = (
    df["Curricular units 1st sem (grade)"] /
    df["Curricular units 1st sem (approved)"]
).fillna(0)

# 4. Financial risk flag: debtor OR tuition not up to date
df["financial_risk"] = (
    (df["Debtor"] == 1) | (df["Tuition fees up to date"] == 0)
).astype(int)

new_features = ["approval_rate_1st_sem", "failed_courses_1st_sem", "avg_first_sem_grade", "financial_risk"]
print(df[new_features].head())

## Data Encoding and Final Cleaning

Drop redundant columns and map the `Target` to numbers (Dropout=0, Enrolled=1, Graduate=2).

In [ ]:
# Drop low-value columns and the EDA-only Is_Dropout flag
columns_to_drop = ['Application mode', 'Nacionality', 'Application order', 'Is_Dropout']
existing_cols_to_drop = [col for col in columns_to_drop if col in df.columns]
df_prepared = df.drop(columns=existing_cols_to_drop)

target_mapping = {'Dropout': 0, 'Enrolled': 1, 'Graduate': 2}
df_prepared['Target'] = df_prepared['Target'].map(target_mapping)

print("Columns removed:", existing_cols_to_drop)
print("Target distribution:\n", df_prepared['Target'].value_counts())
display(df_prepared.head())

## Feature Engineering (Round 2): Trends, Overall Success, and Interactions

All engineered features are created **here, before RFE**, so feature selection can see them.
This fixes the earlier ordering issue where features were added after RFE had already run.

Includes cross-semester trends, overall-success metrics, and **interaction features** aimed
at the ambiguous Enrolled class (students who are mid-trajectory rather than clearly
succeeding or failing).

In [ ]:
# --- Cross-semester trend & overall-success features ---
df_prepared["approved_units_diff"] = (
    df_prepared["Curricular units 2nd sem (approved)"] - df_prepared["Curricular units 1st sem (approved)"]
)

total_enrolled = df_prepared["Curricular units 1st sem (enrolled)"] + df_prepared["Curricular units 2nd sem (enrolled)"]
total_approved = df_prepared["Curricular units 1st sem (approved)"] + df_prepared["Curricular units 2nd sem (approved)"]
df_prepared["overall_approval_rate"] = (total_approved / total_enrolled).fillna(0)

sum_grades_1st = df_prepared["avg_first_sem_grade"] * df_prepared["Curricular units 1st sem (approved)"]
sum_grades_2nd = df_prepared["Curricular units 2nd sem (grade)"] * df_prepared["Curricular units 2nd sem (approved)"]
df_prepared["overall_grade_avg"] = ((sum_grades_1st + sum_grades_2nd) / total_approved).fillna(0)

df_prepared["second_sem_pass_rate"] = (
    df_prepared["Curricular units 2nd sem (approved)"] / df_prepared["Curricular units 2nd sem (enrolled)"]
).fillna(0)

df_prepared["grade_improvement"] = (
    df_prepared["Curricular units 2nd sem (grade)"] - df_prepared["Curricular units 1st sem (grade)"]
).fillna(0)

df_prepared["failures_2nd_sem"] = (
    df_prepared["Curricular units 2nd sem (enrolled)"] - df_prepared["Curricular units 2nd sem (approved)"]
)

df_prepared["missed_eval_ratio_1st"] = (
    df_prepared["Curricular units 1st sem (without evaluations)"] / df_prepared["Curricular units 1st sem (enrolled)"]
).fillna(0)

df_prepared["missed_eval_ratio_2nd"] = (
    df_prepared["Curricular units 2nd sem (without evaluations)"] / df_prepared["Curricular units 2nd sem (enrolled)"]
).fillna(0)

df_prepared["financial_risk_score"] = df_prepared["Debtor"] + (1 - df_prepared["Tuition fees up to date"])

df_prepared["parent_education_avg"] = (
    (df_prepared["Mother's qualification"] + df_prepared["Father's qualification"]) / 2
).fillna(0)

# --- Interaction features (target the ambiguous Enrolled class) ---
# Compounding academic risk: failing courses AND a low pass rate
df_prepared["academic_risk"] = (
    df_prepared["failed_courses_1st_sem"] * (1 - df_prepared["approval_rate_1st_sem"])
)

# Financial stress compounded with academic struggle
df_prepared["financial_academic_risk"] = (
    df_prepared["financial_risk_score"] * df_prepared["failed_courses_1st_sem"]
)

# Grade volatility between semesters (mid-trajectory students swing more)
df_prepared["grade_consistency"] = (
    df_prepared["Curricular units 2nd sem (grade)"] - df_prepared["Curricular units 1st sem (grade)"]
).abs()

engineered = [
    "approved_units_diff", "overall_approval_rate", "overall_grade_avg", "second_sem_pass_rate",
    "grade_improvement", "failures_2nd_sem", "missed_eval_ratio_1st", "missed_eval_ratio_2nd",
    "financial_risk_score", "parent_education_avg",
    "academic_risk", "financial_academic_risk", "grade_consistency",
]
print("Engineered feature count:", len(engineered))
display(df_prepared[engineered].head())

## Feature Scaling

Standardize features with `StandardScaler` (needed for clustering).

In [ ]:
from sklearn.preprocessing import StandardScaler

X = df_prepared.drop(columns=['Target'])
y = df_prepared['Target']

scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X)
df_scaled = pd.DataFrame(X_scaled_array, columns=X.columns)

print("Scaled feature shape:", df_scaled.shape)
display(df_scaled.head())

## K-Means Clustering (Feature Creation)

Group students by similarity and add the cluster label as a synthetic feature that captures non-linear structure.

In [ ]:
from sklearn.cluster import KMeans

inertia = []
k_range = range(1, 11)
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(df_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia, marker='o', linestyle='--')
plt.title('Elbow Method for Optimal Clusters')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.xticks(k_range)
plt.grid(True)
plt.show()

In [ ]:
# k=5 chosen from elbow plot + project context
k_optimal = 5
kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
df_prepared['Cluster'] = kmeans.fit_predict(df_scaled)

print(f'Added Cluster feature with {k_optimal} clusters.')
print(df_prepared['Cluster'].value_counts())

## Recursive Feature Elimination (RFE)

Single RFECV pass with an XGBoost estimator over the **full** feature set (originals +
all engineered + interaction + cluster), so selection reflects every feature.

In [ ]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold

X = df_prepared.drop(columns=['Target'])
y = df_prepared['Target']

estimator = XGBClassifier(
    objective='multi:softmax',
    num_class=len(y.unique()),
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42
)

rfecv = RFECV(
    estimator=estimator,
    step=1,
    cv=StratifiedKFold(5),
    scoring='accuracy',
    min_features_to_select=1,
    n_jobs=-1
)
rfecv.fit(X, y)

print(f"Optimal number of features: {rfecv.n_features_}")
selected_features = X.columns[rfecv.support_]
print("Selected features:", selected_features.tolist())

# Highlight which engineered/interaction features survived selection
kept_engineered = [f for f in selected_features if f in engineered]
print("\nEngineered features kept by RFE:", kept_engineered)

plt.figure(figsize=(10, 6))
plt.xlabel("Number of features selected")
plt.ylabel("Cross validation score (accuracy)")
plt.plot(range(1, len(rfecv.cv_results_['mean_test_score']) + 1), rfecv.cv_results_['mean_test_score'])
plt.title('RFECV: Optimal number of features by accuracy')
plt.grid(True)
plt.show()

X_rfe = X[selected_features].copy()
print("Shape after RFE:", X_rfe.shape)

## Train-Test Split

Single stratified 80/20 split to preserve class balance.

In [ ]:
from sklearn.model_selection import train_test_split

X_final = X_rfe
y_final = y

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final, test_size=0.2, random_state=42, stratify=y_final
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

## Model Training: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("Random Forest Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

## Model Training: XGBoost

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

xgb_model = XGBClassifier(
    objective='multi:softmax',
    num_class=len(y_train.unique()),
    eval_metric='mlogloss',
    random_state=42,
    use_label_encoder=False
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

print("XGBoost Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_xgb))

## Model Comparison: Random Forest vs XGBoost

In [ ]:
from sklearn.metrics import f1_score

model_predictions = {
    "Random Forest": y_pred_rf,
    "XGBoost": y_pred_xgb,
}

model_performance = []
for name, preds in model_predictions.items():
    model_performance.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'F1-Score (Weighted)': f1_score(y_test, preds, average='weighted')
    })

performance_df = pd.DataFrame(model_performance)
performance_melted = performance_df.melt(id_vars='Model', var_name='Metric', value_name='Score')

plt.figure(figsize=(8, 6))
sns.barplot(x='Model', y='Score', hue='Metric', data=performance_melted, palette='viridis')
plt.title('Random Forest vs XGBoost')
plt.ylim(0, 1)
plt.legend(title='Metric')
plt.tight_layout()
plt.show()

display(performance_df.sort_values(by='Accuracy', ascending=False))

## Model Evaluation: ROC Curves and AUC

One-vs-rest ROC curves for Random Forest and XGBoost across the three classes.

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

y_pred_proba_rf = rf_model.predict_proba(X_test)
y_pred_proba_xgb = xgb_model.predict_proba(X_test)

y_test_binarized = label_binarize(y_test, classes=[0, 1, 2])
n_classes = y_test_binarized.shape[1]

model_probs = {'Random Forest': y_pred_proba_rf, 'XGBoost': y_pred_proba_xgb}
roc_data = {name: {'fpr': {}, 'tpr': {}, 'auc': {}} for name in model_probs}

for name, proba in model_probs.items():
    for i in range(n_classes):
        fpr, tpr, _ = roc_curve(y_test_binarized[:, i], proba[:, i])
        roc_data[name]['fpr'][i] = fpr
        roc_data[name]['tpr'][i] = tpr
        roc_data[name]['auc'][i] = auc(fpr, tpr)

class_labels = ['Dropout (0)', 'Enrolled (1)', 'Graduate (2)']
colors = ['blue', 'red', 'green']
linestyles = ['-', '--']

plt.figure(figsize=(15, 7))
for i in range(n_classes):
    plt.subplot(1, n_classes, i + 1)
    for j, (name, data) in enumerate(roc_data.items()):
        plt.plot(data['fpr'][i], data['tpr'][i], color=colors[i], linestyle=linestyles[j],
                 label=f'{name} (AUC = {data["auc"][i]:.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
    plt.title(f'ROC: {class_labels[i]} (OvR)')
    plt.legend(loc='lower right'); plt.grid(True)
plt.tight_layout()
plt.show()

## Hyperparameter Tuning: Random Forest

In [ ]:
from sklearn.model_selection import GridSearchCV
import time

param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'random_state': [42]
}

grid_search_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid_rf,
    cv=3, scoring='accuracy', n_jobs=-1, verbose=2
)

print("Tuning Random Forest...")
start = time.time()
grid_search_rf.fit(X_train, y_train)
print(f"Done in {time.time() - start:.2f}s.")

best_params_rf = grid_search_rf.best_params_
print(f"\nBest params: {best_params_rf}")
print(f"Best CV accuracy: {grid_search_rf.best_score_:.4f}")

In [ ]:
tuned_rf_model = RandomForestClassifier(**best_params_rf)
tuned_rf_model.fit(X_train, y_train)
y_pred_tuned_rf = tuned_rf_model.predict(X_test)

print("Tuned Random Forest Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_tuned_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_tuned_rf))

## Hyperparameter Tuning: XGBoost

In [ ]:
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'random_state': [42]
}

grid_search_xgb = GridSearchCV(
    estimator=XGBClassifier(
        objective='multi:softmax',
        num_class=len(y_train.unique()),
        eval_metric='mlogloss',
        use_label_encoder=False,
        random_state=42
    ),
    param_grid=param_grid_xgb,
    cv=3, scoring='accuracy', n_jobs=-1, verbose=2
)

print("Tuning XGBoost...")
start = time.time()
grid_search_xgb.fit(X_train, y_train)
print(f"Done in {time.time() - start:.2f}s.")

best_params_xgb = grid_search_xgb.best_params_
print(f"\nBest params: {best_params_xgb}")
print(f"Best CV accuracy: {grid_search_xgb.best_score_:.4f}")

In [ ]:
tuned_xgb_model = XGBClassifier(**best_params_xgb)
tuned_xgb_model.fit(X_train, y_train)
y_pred_tuned_xgb = tuned_xgb_model.predict(X_test)

print("Tuned XGBoost Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_tuned_xgb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_tuned_xgb))

# Improving Enrolled-Class Detection

The Enrolled class (1) is the minority and the hardest to classify. Below we test
class-balancing strategies and rank models by **macro-F1 and Enrolled recall/F1**,
not just overall accuracy — a model that catches more at-risk/in-progress students is
more useful for early intervention even if total accuracy dips slightly.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Helper: score a model's predictions with Enrolled-focused metrics
def score_model(name, y_true, y_pred):
    report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Macro F1': f1_score(y_true, y_pred, average='macro'),
        'Enrolled Recall': report['1']['recall'],
        'Enrolled F1': report['1']['f1-score'],
    }

results = []
results.append(score_model("RF (baseline)", y_test, y_pred_rf))
results.append(score_model("RF (tuned)", y_test, y_pred_tuned_rf))
results.append(score_model("XGBoost (tuned)", y_test, y_pred_tuned_xgb))
print("Baselines scored.")

### 1. Random Forest + `class_weight='balanced'`

Reweights classes inversely to frequency so the minority Enrolled class counts more.

In [ ]:
# Reuse tuned RF params, add balanced class weights
rf_balanced = RandomForestClassifier(**{**best_params_rf, 'class_weight': 'balanced'})
rf_balanced.fit(X_train, y_train)
y_pred_rf_balanced = rf_balanced.predict(X_test)

print("RF (balanced) Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_rf_balanced))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf_balanced, zero_division=0))

results.append(score_model("RF (class_weight)", y_test, y_pred_rf_balanced))

### 2. Random Forest + SMOTE

Synthetically oversample the Enrolled class **in the training set only** (no test leakage),
then retrain the tuned Random Forest.

In [ ]:
import sys
!{sys.executable} -m pip install imbalanced-learn
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)
print("Class balance after SMOTE:\n", pd.Series(y_train_smote).value_counts())

In [ ]:
rf_smote = RandomForestClassifier(**best_params_rf)
rf_smote.fit(X_train_smote, y_train_smote)
y_pred_rf_smote = rf_smote.predict(X_test)

print("RF (SMOTE) Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_rf_smote))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf_smote, zero_division=0))

results.append(score_model("RF (SMOTE)", y_test, y_pred_rf_smote))

### 3. Tuned XGBoost + class weights

XGBoost has no `class_weight`; instead we pass balanced **per-sample** weights computed
from the training labels.

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

xgb_balanced = XGBClassifier(**best_params_xgb)
xgb_balanced.fit(X_train, y_train, sample_weight=sample_weights)
y_pred_xgb_balanced = xgb_balanced.predict(X_test)

print("XGBoost (balanced) Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb_balanced))
print("\nClassification Report:\n", classification_report(y_test, y_pred_xgb_balanced, zero_division=0))

results.append(score_model("XGBoost (sample_weight)", y_test, y_pred_xgb_balanced))

### Comparison: Which model best detects Enrolled students?

Ranked by **macro-F1 then Enrolled recall**. The baseline maximizes accuracy; the balanced
variants trade a little accuracy for substantially better Enrolled detection.

In [ ]:
results_df = pd.DataFrame(results).sort_values(
    by=['Macro F1', 'Enrolled Recall'], ascending=False
).reset_index(drop=True)
display(results_df.round(3))

# Visualize the Enrolled-focused metrics
plot_df = results_df.melt(
    id_vars='Model',
    value_vars=['Enrolled Recall', 'Enrolled F1', 'Macro F1', 'Accuracy'],
    var_name='Metric', value_name='Score'
)
plt.figure(figsize=(14, 7))
sns.barplot(x='Model', y='Score', hue='Metric', data=plot_df, palette='viridis')
plt.title('Model Comparison (Enrolled-focused metrics)')
plt.ylim(0, 1)
plt.xticks(rotation=30, ha='right')
plt.legend(title='Metric', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Select the best model by macro-F1 then Enrolled recall
best_row = results_df.iloc[0]
best_name = best_row['Model']

prediction_lookup = {
    "RF (baseline)": (rf_model, y_pred_rf),
    "RF (tuned)": (tuned_rf_model, y_pred_tuned_rf),
    "XGBoost (tuned)": (tuned_xgb_model, y_pred_tuned_xgb),
    "RF (class_weight)": (rf_balanced, y_pred_rf_balanced),
    "RF (SMOTE)": (rf_smote, y_pred_rf_smote),
    "XGBoost (sample_weight)": (xgb_balanced, y_pred_xgb_balanced),
}
best_model, y_pred_best = prediction_lookup[best_name]

print(f"Best model by macro-F1 + Enrolled recall: {best_name}")
print(best_row.round(3).to_string())

## Feature Importance (Random Forest)

Confirms whether the engineered/interaction features rose into the top predictors.

In [ ]:
feature_importances = rf_model.feature_importances_
features_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=features_df)
plt.title('Random Forest Feature Importances')
plt.tight_layout()
plt.show()

print("Top 10 features:")
display(features_df.head(10))

### Top 5 Most Important Features Influencing Dropout

In [ ]:
top_5_features = features_df.head(5)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=top_5_features, palette='viridis')
plt.title('Top 5 Features Influencing Dropout (Random Forest)', fontsize=14)
plt.tight_layout()
plt.show()

display(top_5_features)

## Confusion Matrix: Best Model

Compare the baseline RF confusion matrix with the selected best model — watch the
**Enrolled (row 1)** correct count improve.

In [ ]:
from sklearn.metrics import confusion_matrix

class_names_cm = ['Dropout', 'Enrolled', 'Graduate']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (title, preds) in zip(axes, [("RF (baseline)", y_pred_rf), (f"Best: {best_name}", y_pred_best)]):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names_cm, yticklabels=class_names_cm, ax=ax)
    ax.set_title(f'Confusion Matrix: {title}')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.tight_layout()
plt.show()

## Interpret Predictions with SHAP

Use SHAP to explain individual student predictions and per-class feature impact for the tuned Random Forest.

In [ ]:
import sys
!{sys.executable} -m pip install shap
import shap

# TreeExplainer on the tuned Random Forest
explainer = shap.TreeExplainer(tuned_rf_model, feature_names=X_train.columns)
print("SHAP explainer initialized.")

In [ ]:
# Sample a few students to explain individually
sample_X_test = X_test.sample(n=5, random_state=42)
shap_values = explainer.shap_values(sample_X_test)
print(f"Computed SHAP values for {sample_X_test.shape[0]} students.")
display(sample_X_test.head())

In [ ]:
# Waterfall plot for one student's predicted class
class_names = ['Dropout', 'Enrolled', 'Graduate']
instance_idx = 0
sample_instance = sample_X_test.iloc[[instance_idx]]
predicted_class = tuned_rf_model.predict(sample_instance)[0]
print(f"Student {sample_instance.index[0]} predicted as '{class_names[predicted_class]}'")

# TreeExplainer multi-class output: shape (n_samples, n_features, n_classes)
shap_values_for_instance = shap_values[instance_idx, :, predicted_class]

explanation = shap.Explanation(
    values=shap_values_for_instance,
    base_values=explainer.expected_value[predicted_class],
    data=sample_X_test.iloc[instance_idx].values,
    feature_names=X_train.columns.tolist()
)

shap.waterfall_plot(explanation, max_display=10, show=False)
plt.title(f'SHAP Waterfall: Student {sample_instance.index[0]} (Predicted: {class_names[predicted_class]})')
plt.tight_layout()
plt.show()

In [ ]:
# Global feature impact across classes
shap.summary_plot(shap_values, sample_X_test, plot_type="violin", class_names=class_names)

In [ ]:
# Per-class SHAP feature importance for the 'Enrolled' class (class 1)
shap_values_full = explainer.shap_values(X_test)
class_to_analyze = 1
shap_values_enrolled = shap_values_full[:, :, class_to_analyze]
mean_abs_shap_enrolled = np.abs(shap_values_enrolled).mean(axis=0)

feature_importance_df_enrolled = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': mean_abs_shap_enrolled
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df_enrolled.head(10), palette='viridis')
plt.title(f'Top 10 Features Influencing "{class_names[class_to_analyze]}" (Mean |SHAP|)', fontsize=14)
plt.tight_layout()
plt.show()

display(feature_importance_df_enrolled.head(10))

## Conclusion: Key Drivers and Action Plan

**Improving the Enrolled class.** Enrolled is the minority class and genuinely ambiguous —
these students share traits with both future graduates and future dropouts. We addressed
it by (1) engineering all features before a single RFE pass so selection sees trend and
interaction terms, and (2) testing class-balancing methods (RF `class_weight='balanced'`,
RF + SMOTE, XGBoost balanced sample weights). Models were ranked by macro-F1 and Enrolled
recall/F1 rather than raw accuracy, since catching at-risk/in-progress students early is
the project's actual goal. The comparison table above identifies the best trade-off model.

**Top features influencing student success:**

- **Overall Approval Rate** — units approved across both semesters; the single strongest signal of consistent academic performance.
- **Second Semester Pass Rate** — proportion of 2nd-sem units passed; strong indicator of continued success.
- **Second Semester Failures** — count of 2nd-sem units failed; significant predictor of negative outcomes.
- **First Semester Approval Rate** — early-warning sign when low.
- **Curricular Units 2nd Sem (Approved)** — raw count of completed 2nd-sem units.

**Three plans of action:**

1. **Early Warning & Targeted Intervention** — monitor `approval_rate_1st_sem` and early failures by mid-first-semester; flag low performers for mandatory tutoring, counseling, and personalized success plans.
2. **Second-Semester Reinforcement & Retention** — build on first-semester support with time-management, study-skill, and stress workshops; tie `second_sem_pass_rate` and `failures_2nd_sem` to one-on-one coaching and peer/faculty mentorship.
3. **Holistic Well-being & Resource Integration** — a multi-department task force (advising, mental health, financial aid, career services) to address non-academic barriers for students flagged by the early-warning system.

**Next steps:** if more time is available, tune the SMOTE sampling ratio, try a cascading
classifier (Dropout vs Not-Dropout, then Graduate vs Enrolled), and re-run RFE per balanced
model since the optimal feature subset can shift after resampling.